## Preprocesamiento con scikit-learn

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

#Seleccionar variables relevantes
vars_num = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
            'fico_range_high', 'emp_length']

vars_cat = ['purpose', 'home_ownership', 'addr_state', 'verification_status']

TARGET = 'default'

df_model = df_eda[vars_num + vars_cat + [TARGET]].copy()

X = df_model[vars_num + vars_cat]
y = df_model[TARGET]

#División train_test_split (80/20), estratificada por clase
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("  DIVISIÓN TRAIN_TEST_SPLIT \n")
print(f"  Train : {X_train.shape[0]:>10,} filas  "
      f"({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Test  : {X_test.shape[0]:>10,} filas  "
      f"({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\n  Distribución target en train:")
print((y_train.value_counts(normalize=True)*100).round(2).to_string())
print(f"\n  Distribución target en test:")
print((y_test.value_counts(normalize=True)*100).round(2).to_string())

#Codificación de variables categóricas (OneHotEncoder o LabelEncoder)
preprocessor = ColumnTransformer(
    transformers=[
        ('num',
         Pipeline([
             ('imputer', SimpleImputer(strategy='median')),
             ('scaler',  StandardScaler())
         ]),
         vars_num),

        ('cat',
         Pipeline([
             ('imputer', SimpleImputer(strategy='most_frequent')),
             ('encoder', OneHotEncoder(
                 drop='first',
                 handle_unknown='ignore',
                 sparse_output=False
             ))
         ]),
         vars_cat)
    ],
    remainder='drop'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

cat_feature_names = preprocessor.named_transformers_['cat']['encoder']\
                                 .get_feature_names_out(vars_cat)
all_feature_names = vars_num + list(cat_feature_names)

print("\n PREPROCESAMIENTO")
print("=" * 60)
print(f"  Shape X_train original  : {X_train.shape}")
print(f"  Shape X_train procesado : {X_train_proc.shape}")
print(f"  Shape X_test  procesado : {X_test_proc.shape}")

# Verificar que no quedan nulos tras preprocesamiento
nulos_train = np.isnan(X_train_proc).sum()
nulos_test  = np.isnan(X_test_proc).sum()
print(f"\n  Nulos en X_train procesado : {nulos_train}")
print(f"  Nulos en X_test  procesado : {nulos_test}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import roc_auc_score
import time

In [ ]:
param_grid = {'n_estimators': [10, 50, 100], 'max_depth': [5, 10, 15]}
random_forest = RandomForestClassifier(random_state=42)

# Entrenamiento con GridSearchCV
start_train = time.time()
grid = GridSearchCV(random_forest, param_grid=param_grid, scoring='roc_auc', cv=3)
grid.fit(X_train_proc, y_train)
elapsed_train = time.time() - start_train

# Predicción
start_pred = time.time()
y_pred = grid.predict(X_test_proc)
y_prob = grid.predict_proba(X_test_proc)[:, 1]
elapsed_pred = time.time() - start_pred

# Métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("\nGridSearchCV (scikit-learn) — RandomForestClassifier")
print("Mejores parámetros:", grid.best_params_)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC AUC:   {auc:.4f}")
print("\nMatriz de confusión:")
print(cm)
print(f"\nTiempo de entrenamiento: {elapsed_train:.2f} segundos")
print(f"Tiempo de predicción:    {elapsed_pred:.4f} segundos")

## Preprocesamiento con PySpark

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Imputer
from pyspark.ml.classification import RandomForestClassifier as SparkRFC
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel
import time

# ── 1. Configuración de Spark ────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("LendingClub_Optimized") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.memory.fraction", 0.8) \
    .config("spark.memory.storageFraction", 0.3) \
    .getOrCreate()

In [ ]:
# ── 2. Lectura del CSV ───────────────────────────────────────────────────────
df = spark.read.csv(
    'data/lending_club/accepted_2007_to_2018Q4.csv.gz',
    header=True,
    inferSchema=True
)

# ── 3. Crear variable target binaria ─────────────────────────────────────────
# 0 = Fully Paid | 1 = Charged Off | resto → null (se descarta)
df = df.withColumn(
    "label",
    F.when(F.col("loan_status") == "Fully Paid",   F.lit(0.0))
     .when(F.col("loan_status") == "Charged Off",  F.lit(1.0))
     .otherwise(None)
).filter(F.col("label").isNotNull())

# ── 4. Definir columnas ──────────────────────────────────────────────────────
categorical_cols = ["grade", "sub_grade", "home_ownership",
                    "verification_status", "purpose", "addr_state"]

numerical_cols   = ["loan_amnt", "int_rate", "annual_inc",
                    "dti", "fico_range_high", "emp_length"]

In [ ]:
# ── 5. Limpieza de columnas numéricas ────────────────────────────────────────
# Columnas con caracteres no numéricos o leídas como string
for c in ["annual_inc", "dti", "fico_range_high"]:
    df = df.withColumn(
        c,
        F.expr(f"try_cast(regexp_replace(`{c}`, '[^0-9.]', '') AS DOUBLE)")
    )

# emp_length: "10+ years" → 10.0 | "< 1 year" → 0.5 | null/'' → null
df = df.withColumn(
    "emp_length",
    F.when(
        F.col("emp_length").isNull() | (F.trim(F.col("emp_length")) == ''),
        None
    ).when(
        F.col("emp_length").contains("< 1"), F.lit(0.5)
    ).otherwise(
        F.expr("try_cast(regexp_extract(`emp_length`, '(\\\\d+)', 1) AS DOUBLE)")
    )
)

# ── 6. Imputar nulls en numéricas con la media (distribuido, sin toPandas) ───
imputer = Imputer(
    inputCols  = numerical_cols,
    outputCols = numerical_cols,
    strategy   = "mean"
)
df = imputer.fit(df).transform(df)

# ── 7. Limpieza de nulls en columnas categóricas ─────────────────────────────
for c in categorical_cols:
    df = df.withColumn(
        c,
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ''), F.lit("unknown"))
         .otherwise(F.col(c))
    )

# ── 8. StringIndexer para categóricas ────────────────────────────────────────
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

# ── 9. OneHotEncoder ─────────────────────────────────────────────────────────
encoder = OneHotEncoder(
    inputCols  = [f"{c}_idx" for c in categorical_cols],
    outputCols = [f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

# ── 10. VectorAssembler ──────────────────────────────────────────────────────
assembler = VectorAssembler(
    inputCols = numerical_cols + [f"{c}_ohe" for c in categorical_cols],
    outputCol = "features_raw",
    handleInvalid="keep"
)

# ── 11. StandardScaler ───────────────────────────────────────────────────────
scaler = StandardScaler(
    inputCol = "features_raw",
    outputCol = "features",
    withMean  = False,   # False porque OHE genera vectores sparse
    withStd   = True
)

# ── 12. Aplicar transformaciones ─────────────────────────────────────────────
df_indexed = df
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

df_encoded   = encoder.fit(df_indexed).transform(df_indexed)
df_assembled = assembler.transform(df_encoded)
df_scaled    = scaler.fit(df_assembled).transform(df_assembled)

# ── 13. Selección + Cache obligatorio ────────────────────────────────────────
df_final = df_scaled.select("features", "label")
df_final.persist(StorageLevel.MEMORY_AND_DISK)
df_final.count()                             # materializa el cache

# ── 14. División estratificada 80/20 ─────────────────────────────────────────
# sampleBy evita toPandas — trae solo 2 filas de metadatos al driver
train_df = df_final.sampleBy("label", fractions={0.0: 0.8, 1.0: 0.8}, seed=42)
test_df  = df_final.subtract(train_df)